In [2]:
# %load_ext autoreload
# %autoreload 3
import torch
import os
import tiny_trainer
from tiny_trainer import *
from video_model import DiTModelWrapper
from videogpt.data import preprocess
from dataclasses import dataclass
import wandb
import torch.distributed as dist
import hydra
from omegaconf import DictConfig, OmegaConf
from hydra import compose, initialize
import os
from torch.utils.data import Dataset
import torchvision
import numpy as np
import torch
import torch
from typing import *
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from torch.distributed.optim import ZeroRedundancyOptimizer
from streaming.vision.base import StreamingDataset
import sys
sys.path.insert(0, "/home/nchen3/video_generation/TATS/") # go to parent dir
import tats
from tats.tats_vqgan import VQGAN
from torch.utils.data.distributed import DistributedSampler

In [3]:
vqgan = VQGAN.load_from_checkpoint('../TATS/vqgan_sky_128_488_epoch=12-step=29999-train.ckpt')
vocab_size = vqgan.codebook.embeddings.shape[0]
input_dim = vqgan.codebook.embeddings.shape[1]
vqgan = None

/home/nchen3/miniconda3/envs/rl/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/nchen3/miniconda3/envs/rl/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


loaded pretrained LPIPS loss from /home/nchen3/video_generation/TATS/tats/modules/cache/vgg.pth


In [9]:
import torch.nn.functional as F
from tats.utils import shift_dim
def get_encoding(self, encodings):
    return F.embedding(encodings, self.codebook.embeddings)
# def decode(self, encodings):
#     h = F.embedding(encodings, self.codebook.embeddings)
#     h = self.post_vq_conv(shift_dim(h, -1, 1))
#     return self.decoder(h)
def decode_cont(self, h):
    h = self.post_vq_conv(shift_dim(h, -1, 1))
    return self.decoder(h)

In [15]:
dataset = StreamingDataset(
            local="../datasets/sky_128_vqgan/", 
            remote=None, 
            split=None,
            shuffle=True,
            shuffle_algo="naive",
            num_canonical_nodes=1,
            batch_size = 1)
dataset[0]

Because `predownload` was not specified, it will default to 8*batch_size if batch_size is not None, otherwise 64. Prior to Streaming v0.7.0, `predownload` defaulted to max(batch_size, 256 * batch_size // num_canonical_nodes).


{'video': array([13688, 14333, 14776, ...,  7504, 15217,  7504], dtype=uint16),
 'video_name': '07U1fSrk9oI_1'}

In [28]:
.shape

(16384, 256)

In [63]:
import numpy as np
class SingleDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, base_l):
        self.dataset = dataset
        self.base_l = base_l
        self.l = len(self.dataset)
    def __len__(self):
        return self.l
    def __getitem__(self, i):
        return self.dataset[i %self.base_l]

class VQVAELookup(torch.utils.data.Dataset):
    def __init__(self, vqvae, dataset):
        self.dataset = dataset
        self.vqvae = vqgan.codebook.embeddings.numpy()
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, i):
        d = self.dataset[i]
        z = self.vqvae[d['video']]
        return {'video' : z.reshape((4, 16, 16, input_dim)).transpose((3,0,1,2)), 'video_name' : d['video_name']}

In [ ]:
from typing import *
@dataclass
class VideoDatasetInfo:
    is_latent : bool
    image_shape : Tuple
    vae_preprocessor : Any
    vae_postprocessor : Any

n_layer = 12
n_embd = 512
n_inner = n_embd * 4
n_head = 16
class MNISTFactory(AbstractTrainerFactory):
    def __init__(self, world_size, per_device_batch_size, gradient_accum_step,use_single=False):
        self.per_device_batch_size = per_device_batch_size
        self.world_size = world_size
        self.gradient_accum_step=gradient_accum_step
        self.use_single = use_single
    def make_model(self):
        model = DiTModelWrapper(
            num_attention_heads = n_head,
            attention_head_dim = n_embd // n_head,
            in_channels = input_dim,
            out_channels = input_dim,
            num_layers = n_layer,
            dropout = 0.0,
            norm_num_groups = 16,
            attention_bias = True,
            spatial_size = 16,
            temporal_size = 4,
            spatial_patch_size = 2,
            temporal_patch_size = 1,
            num_embeds_ada_norm = None,
            class_condition=False
        )
        return model
    def make_optimizer(self, model):
        return ZeroRedundancyOptimizer(model.parameters(), optimizer_class=torch.optim.AdamW, lr = 3e-4, weight_decay=0.0)
    def make_dataloader(self, rank : int): 
        per_device_batch_size = self.per_device_batch_size
        vqgan = VQGAN.load_from_checkpoint('../TATS/vqgan_sky_128_488_epoch=12-step=29999-train.ckpt').to(rank).eval()
        dataset = StreamingDataset(
            local="../datasets/sky_128_vqgan/", 
            remote=None, 
            split=None,
            shuffle=True,
            shuffle_algo="naive",
            num_canonical_nodes=1,
            batch_size = per_device_batch_size)
        dataset = VQVAELookup(vqgan, dataset)
        if self.use_single:
            dataset = SingleDataset(dataset, base_l=64)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=per_device_batch_size, drop_last=True)
        if self.world_size == 1 or self.use_single:
            sampler = None
        else:
            sampler = DistributedSampler(dataset, num_replicas=self.world_size, rank=rank, shuffle=True, seed = 0, drop_last=True)
        return VideoDatasetInfo(image_shape=(input_dim, 4, 16, 16), 
                                is_latent=True,
                               vae_preprocessor=lambda x : decode_cont(vqgan, x.permute((0,2,3,4,1))),
                               vae_postprocessor=None), dataloader, sampler
    def make_scheduler(self, optimizer):
        return torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=500) 

from tiny_trainer import register_config, MISSING
from dataclasses import dataclass
@register_config(group="diffusion", name="base")
@dataclass
class DiffusionConfig:
    sample_type : str = "lognorm0_1"
    max_timestep : int = 512
    sampling_step : int = 50

@torch.no_grad
def sample_image(rank : int, model, max_timestep: int, sampling_step : int, batch_size, class_labels, datasetinfo, use_classlabel, device):
    image_shape = datasetinfo.image_shape
    # we add a generator to better monitor the quality here
    X0 = torch.randn((batch_size, *image_shape)).to(device)
    for i in range(sampling_step):
        val = (i / sampling_step) * max_timestep
        # we use a uniform sampling here
        timestep = torch.full(size=(batch_size,), fill_value=val).to(device)
        if use_classlabel:
            pred = model(X0, timestep=timestep, class_labels = class_labels)
        else:
            pred = model(X0, timestep=timestep)
        X0 += pred.sample * (1 / sampling_step)
    if datasetinfo.is_latent:
        X0 = datasetinfo.vae_preprocessor(X0)
        # X0 = datasetinfo.vae.decode(X0).sample
    return X0

@torch.no_grad
def display_video_tensor(video, fps=4):
    # Ensure tensor is on CPU
    # B, C, T, H, W
    # 4, 3, 16, 64, 64
    # the input should be in [0, 1] !
    video = torch.clip(video, 0, 1).transpose(1,2)
    # print(video.shape)
    # Ensure the video is in uint8 format
    # assert video.shape == (4, 16, 3, 64, 64)
    video = (video * 255).to(torch.uint8)
    video = video.cpu().detach().numpy()
    # Display the video
    video = wandb.Video(video, fps=fps)
    wandb.log({"video": video}, commit=False)

class MNISTTrainer(DefaultTrainer):
    def __init__(self, diff_config : DiffusionConfig, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.diff_config = diff_config
        self.loss_fun = torch.nn.MSELoss()
    def eval_model(self,i, is_debug = False):
        model = self.model
        model.eval()
        def normalize_image(x):
            return x + 0.5
        datasetinfo = self.datasetinfo
        world_size = self.world_size
        max_timestep = self.diff_config.max_timestep
        rank = self.rank
        sampling_step = self.diff_config.sampling_step
        use_classlabel = True
        device = rank
        total_batch_size = 4
        class_labels = None
        step = total_batch_size // world_size
        batch_size = step
        image_array = sample_image(rank,
                                   model, 
                                   max_timestep,
                                   sampling_step,
                                   batch_size = batch_size, 
                                   class_labels = class_labels,
                                   datasetinfo = datasetinfo,
                                   use_classlabel = use_classlabel,
                                   device= device)
        
        image_arrays = [torch.zeros_like(image_array, device=rank) for _ in range(world_size)]
        dist.all_gather(image_arrays, image_array)
        if rank == 0:
            image_array = torch.cat(image_arrays, dim = 0)
            if datasetinfo.is_latent:
                image_array = normalize_image(image_array)
            else:
                image_array = normalize_image(image_array)
            display_video_tensor(image_array)
    def prepare_input(self, data):
        image = data["video"].to(self.rank)
        device = self.rank
        # (B, C, T, H, W)
        noise = torch.randn(image.shape).to(device)
        sample_type = self.diff_config.sample_type
        max_timestep = self.diff_config.max_timestep
        if sample_type == "uniform":
            timestep = torch.rand(size=(image.size(0),)).to(device) * max_timestep
        elif sample_type == "lm":
            x = torch.tensor([lmpdf((i + 1/2)/max_timestep) + 0.1 for i in range(max_timestep)]).to(device)
            x = x/x.sum()
            timestep = torch.multinomial(x, image.size(0)).to(torch.float32) # [0, max_timestep-1]
            timestep += torch.rand(size=(image.size(0),)).to(device) # [0, max_timestep)
        elif sample_type == "lognorm0_1":
            x = torch.randn(size=(image.size(0),)).to(device)
            # map [-inf, inf] to [0, 1]
            x = torch.sigmoid(x)
            # scale the timestep into [0, max_timestep]
            timestep = x * max_timestep
        alpha = (timestep / max_timestep).view(-1, *([1]*(len(image.shape) - 1)))
        point = (1 - alpha) * noise + alpha * image
        target = image - noise
        return {"point" : point, "target" : target, "timestep" : timestep}
    def calculate_loss(self, point, timestep, target):
        model = self.model
        output = model(point, timestep=timestep, class_labels = None)
        sample = output.sample
        return self.loss_fun(sample, target)

if __name__ == "__main__":
    with initialize(version_base=None, config_path="config"):
        cfg = compose(config_name="mnist_config_single")
        trainer_cfg = cfg.trainer
        factory = MNISTFactory(world_size = trainer_cfg.world_size, 
                               per_device_batch_size = 32 // trainer_cfg.world_size,
                               gradient_accum_step = 1,
                               use_single=True)
        trainer = MNISTTrainer(cfg.diffusion, factory, trainer_cfg)
        trainer.run()

Because `predownload` was not specified, it will default to 8*batch_size if batch_size is not None, otherwise 64. Prior to Streaming v0.7.0, `predownload` defaulted to max(batch_size, 256 * batch_size // num_canonical_nodes).


loaded pretrained LPIPS loss from /home/nchen3/video_generation/TATS/tats/modules/cache/vgg.pth


wandb: ERROR Error while calling W&B API: failed to find run video_generation_test/xh7v1hgn (<Response [404]>)
wandb: ERROR Error while calling W&B API: run video_generation_test/xh7v1hgn not found during createRunFiles (<Response [404]>)
wandb: ERROR Error while calling W&B API: run video_generation_test/xh7v1hgn not found during createRunFiles (<Response [404]>)
wandb: ERROR Error while calling W&B API: run video_generation_test/xh7v1hgn not found during createRunFiles (<Response [404]>)
wandb: ERROR Error while calling W&B API: run video_generation_test/xh7v1hgn not found during createRunFiles (<Response [404]>)
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)


  0%|          | 0/100000 [00:00<?, ?it/s]

  0%|          | 0/2430 [00:00<?, ?it/s]

In [ ]:
print(tiny_trainer.TrainerConfig())

In [30]:
cfg

{'trainer': {'world_size': 1, 'epoch': 100000, 'gradient_accum_step': 1, 'dtype': 'torch.bfloat16', 'clip_norm': 1.0, 'eval_n_sample': 10000, 'save_n_sample': 100000000, 'compile_model': False, 'enable_wandb': False, 'project_name': 'video_generation_test', 'run_name': 'mnist_test', 'enable_profile': False, 'profiler_config': {'profile_memory': True, 'record_shapes': True, 'with_flops': True, 'with_stack': False, 'skip_first': 5, 'wait': 2, 'warmup': 2, 'active': 5, 'repeat': 1}}, 'diffusion': {'sample_type': 'lognorm0_1', 'max_timestep': 512}}